## Import Library

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

## Load Data

In [3]:
#train = pd.read_csv('sales_train.csv', parse_dates=['date'])
#test = pd.read_csv('sales_test.csv', parse_dates=['date'])
#ss = pd.read_csv('solution.csv')
#inventory = pd.read_csv('inventory.csv')
#weights = pd.read_csv('test_weights.csv')
calendar = pd.read_csv('calendar.csv', parse_dates=['date'])

#### Check data before we do anything
七个warehouse，五个国家
930行有holiday
166个有holiday没有名字

In [4]:
warehouses = calendar["warehouse"].unique()
print(warehouses)
print(calendar["holiday_name"].count())
print(calendar["holiday_name"].unique())
print((calendar["winter_school_holidays"] == 1).sum())
print((calendar["school_holidays"] == 1).sum())
print(calendar[(calendar["holiday"] == 1) & (pd.isna(calendar["holiday_name"]))])

['Frankfurt_1' 'Prague_2' 'Brno_1' 'Munich_1' 'Prague_3' 'Prague_1'
 'Budapest_1']
930
[nan 'Den boje za svobodu a demokracii' 'Good Friday' 'Easter Monday'
 '2nd Christmas Day' 'Cyrila a Metodej' 'International womens day'
 'Den ceske statnosti' 'Den osvobozeni' 'New Years Day' 'Whit sunday'
 'Memorial Day of the Republic' 'Independent Hungary Day' 'Labour Day'
 'Memorial Day for the Victims of the Holocaust' 'Reformation Day'
 'Den vzniku samostatneho ceskoslovenskeho statu' 'Ascension day'
 'Corpus Christi' 'Jan Hus' 'Assumption of the Virgin Mary' 'Epiphany'
 'Christmas Eve' 'Memorial day of the 1956 Revolution'
 'Memorial Day for the Martyrs of Arad' 'Day of National Unity'
 '1st Christmas Day' 'Whit monday' 'German Unity Day'
 'State Foundation Day' 'All Saints Day' 'Hungary National Day Holiday'
 'Christmas Holiday'
 'Memorial Day for the Victims of the Communist Dictatorships'
 'Peace Festival in Augsburg' 'National Defense Day'
 '1848 Revolution Memorial Day (Extra holiday)' "

## 补充节日

In [5]:
# Holidays get from https://www.holidays-info.com/
# https://www.holidays-info.com/czech-republic/calendar/prague/2024/


czech_holidays = [  # Prague
    (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020','04/21/2019', '04/01/2018', '04/16/2017', '03/27/2016'], 'Easter Day'), 
    (['04/01/2024', '04/10/2023', '04/18/2022', '04/05/2021', '04/13/2020','04/22/2019', '04/02/2018', '04/17/2017', '03/28/2016'], 'Easter Monday'),
    (['05/12/2024', '05/14/2023', '05/08/2022', '05/09/2021', '05/10/2020','05/12/2019', '05/13/2018', '05/14/2017', '05/08/2016'], "Mother's Day"), 
    (['03/30/2024', '04/08/2023', '04/16/2022', '04/03/2021', '04/11/2020','04/20/2019', '03/31/2018', '04/15/2017', '03/26/2016'], 'Holy Saturday'),
    (['03/29/2024', '04/07/2023', '04/15/2022', '04/02/2021', '04/10/2020','04/19/2019', '03/30/2018', '04/14/2017', '03/25/2016'], 'Good Friday'),
]


brno_holidays = [  # Brno
    (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020','04/21/2019', '04/01/2018', '04/16/2017', '03/27/2016'], 'Easter Day'), 
    (['04/01/2024', '04/10/2023', '04/18/2022', '04/05/2021', '04/13/2020','04/22/2019', '04/02/2018', '04/17/2017', '03/28/2016'], 'Easter Monday'),
    (['05/12/2024', '05/14/2023', '05/08/2022', '05/09/2021', '05/10/2020','05/12/2019', '05/13/2018', '05/14/2017', '05/08/2016'], "Mother's Day"), 
    (['03/30/2024', '04/08/2023', '04/16/2022', '04/03/2021', '04/11/2020','04/20/2019', '03/31/2018', '04/15/2017', '03/26/2016'], 'Holy Saturday'),
    (['03/29/2024', '04/07/2023', '04/15/2022', '04/02/2021', '04/10/2020','04/19/2019', '03/30/2018', '04/14/2017', '03/25/2016'], 'Good Friday'),
]

budapest_holidays = [  # Budapest
    (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020','04/21/2019', '04/01/2018', '04/16/2017', '03/27/2016'], 'Easter Day'), 
    (['04/01/2024', '04/10/2023', '04/18/2022', '04/05/2021', '04/13/2020','04/22/2019', '04/02/2018', '04/17/2017', '03/28/2016'], 'Easter Monday'),
    (['05/12/2024', '05/14/2023', '05/08/2022', '05/09/2021', '05/10/2020','05/12/2019', '05/13/2018', '05/14/2017', '05/08/2016'], "Mother's Day"), 
    (['03/30/2024', '04/08/2023', '04/16/2022', '04/03/2021', '04/11/2020','04/20/2019', '03/31/2018', '04/15/2017', '03/26/2016'], 'Holy Saturday'),
    (['03/29/2024', '04/07/2023', '04/15/2022', '04/02/2021', '04/10/2020','04/19/2019', '03/30/2018', '04/14/2017', '03/25/2016'], 'Good Friday'),
]

# Bavaria - Munich
munich_holidays = [
   (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020','04/21/2019', '04/01/2018', '04/16/2017', '03/27/2016'], 'Easter Day'), 
    (['04/01/2024', '04/10/2023', '04/18/2022', '04/05/2021', '04/13/2020','04/22/2019', '04/02/2018', '04/17/2017', '03/28/2016'], 'Easter Monday'),
    (['05/12/2024', '05/14/2023', '05/08/2022', '05/09/2021', '05/10/2020','05/12/2019', '05/13/2018', '05/14/2017', '05/08/2016'], "Mother's Day"), 
    (['03/30/2024', '04/08/2023', '04/16/2022', '04/03/2021', '04/11/2020','04/20/2019', '03/31/2018', '04/15/2017', '03/26/2016'], 'Holy Saturday'),
    (['03/29/2024', '04/07/2023', '04/15/2022', '04/02/2021', '04/10/2020','04/19/2019', '03/30/2018', '04/14/2017', '03/25/2016'], 'Good Friday'),
]

# Hesse - Frankfurt
frank_holidays = [
    (['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020','04/21/2019', '04/01/2018', '04/16/2017', '03/27/2016'], 'Easter Day'), 
    (['04/01/2024', '04/10/2023', '04/18/2022', '04/05/2021', '04/13/2020','04/22/2019', '04/02/2018', '04/17/2017', '03/28/2016'], 'Easter Monday'),
    (['05/12/2024', '05/14/2023', '05/08/2022', '05/09/2021', '05/10/2020','05/12/2019', '05/13/2018', '05/14/2017', '05/08/2016'], "Mother's Day"), 
    (['03/30/2024', '04/08/2023', '04/16/2022', '04/03/2021', '04/11/2020','04/20/2019', '03/31/2018', '04/15/2017', '03/26/2016'], 'Holy Saturday'),
    (['03/29/2024', '04/07/2023', '04/15/2022', '04/02/2021', '04/10/2020','04/19/2019', '03/30/2018', '04/14/2017', '03/25/2016'], 'Good Friday'),
]

# df_fill function
def fill_loss_holidays(df_fill, warehouses, holidays):
    df = df_fill.copy()
    for item in holidays: 
        dates, holiday_name = item
        # 对日期进行格式化操作 12/29/2019 会变成 2019-12-29
        generated_dates = [datetime.strptime(date, '%m/%d/%Y').strftime('%Y-%m-%d') for date in dates]
        for generated_date in generated_dates:
            df.loc[(df['warehouse'].isin(warehouses)) & (df['date'] == generated_date), 'holiday'] = 1
            df.loc[(df['warehouse'].isin(warehouses)) & (df['date'] == generated_date), 'holiday_name'] = holiday_name
    return df

# 给每家店添加节日
calendar = fill_loss_holidays(df_fill=calendar, warehouses=['Prague_1', 'Prague_2', 'Prague_3'], holidays=czech_holidays)
calendar = fill_loss_holidays(df_fill=calendar, warehouses=['Brno_1'], holidays=brno_holidays)
calendar = fill_loss_holidays(df_fill=calendar, warehouses=['Munich_1'], holidays=munich_holidays)
calendar = fill_loss_holidays(df_fill=calendar, warehouses=['Frankfurt_1'], holidays=frank_holidays)
calendar = fill_loss_holidays(df_fill=calendar, warehouses=['Budapest_1'], holidays=budapest_holidays)

# 把有假日名字但没标holiday的补上
calendar.loc[calendar['holiday_name'].notna(), 'holiday'] = 1

# 每家店按照日期排序
calendar = calendar.sort_values(by=['warehouse', 'date'])
calendar.head()


,date,holiday_name,holiday,shops_closed,winter_school_holidays,school_holidays,warehouse
17698,2016-01-01,New Years Day,1,1,0,0,Brno_1
12672,2016-01-02,NaN,0,0,0,0,Brno_1
12440,2016-01-03,NaN,0,0,0,0,Brno_1
7344,2016-01-04,NaN,0,0,0,0,Brno_1
11523,2016-01-05,NaN,0,0,0,0,Brno_1


## 分出年月日星期列

In [6]:
# create new columns for year, month, and day of week based on date
calendar['year'] = calendar['date'].dt.year
calendar['month'] = calendar['date'].dt.month
calendar['day'] = calendar['date'].dt.day
calendar['day_of_week'] = calendar['date'].dt.dayofweek

In [7]:
#验证：所有节日都有名字
print(calendar[(calendar["holiday"] == 1) & (pd.isna(calendar["holiday_name"]))])

Empty DataFrame
Columns: [date, holiday_name, holiday, shops_closed, winter_school_holidays, school_holidays, warehouse, year, month, day, day_of_week]
Index: []


## 添加在某一年出现但是其他年份没有被标记的节日

In [8]:
# Ensure 'date' is in datetime format
calendar['date'] = pd.to_datetime(calendar['date'])

# Extract day and month for correct year-wise comparison
calendar['_day_month'] = calendar['date'].dt.strftime('%m-%d')

# Create a previous year column in the main calendar DataFrame
calendar['year_prev'] = calendar['year'] - 1  # Mapping to the previous year

# Create a reference DataFrame containing previous year holidays per warehouse
holiday_mapping = calendar[['warehouse', 'year', '_day_month', 'holiday', 'holiday_name']].copy()
holiday_mapping.rename(columns={'year': 'year_prev'}, inplace=True)  # Rename for correct merge

# Merge calendar with itself to find holiday matches within the same warehouse and same day of the year
calendar = calendar.merge(
    holiday_mapping,
    on=['warehouse', '_day_month', 'year_prev'],  # Ensuring correct matching
    suffixes=('', '_prev'),
    how='left'
)

# Update holiday column and holiday_name **only when the previous year had a holiday**
mask = (calendar['holiday_prev'].notna()) & (calendar['holiday_name_prev'].notna())

calendar.loc[mask, 'holiday'] = calendar.loc[mask, 'holiday_prev']
calendar.loc[mask, 'holiday_name'] = calendar.loc[mask, 'holiday_name_prev']

# Remove temporary columns to maintain original structure
calendar.drop(columns=['_day_month', 'year_prev', 'holiday_prev', 'holiday_name_prev'], inplace=True)

In [9]:
#export calendar to csv
#calendar.to_csv('calendar_processed.csv', index=False)

# print calendar
print(calendar)

            date   holiday_name  holiday  shops_closed  \
0     2016-01-01  New Years Day        1             1   
1     2016-01-02            NaN        0             0   
2     2016-01-03            NaN        0             0   
3     2016-01-04            NaN        0             0   
4     2016-01-05            NaN        0             0   
...          ...            ...      ...           ...   
23011 2024-12-27            NaN        0             0   
23012 2024-12-28            NaN        0             0   
23013 2024-12-29            NaN        0             0   
23014 2024-12-30            NaN        0             0   
23015 2024-12-31            NaN        0             0   

       winter_school_holidays  school_holidays warehouse  year  month  day  \
0                           0                0    Brno_1  2016      1    1   
1                           0                0    Brno_1  2016      1    2   
2                           0                0    Brno_1  2016      1

## cld data cleaning basics
√小写所有字母
√去名字空格
√每个节日有单独column

In [10]:
# Step 1: Clean the `holiday_name` column
calendar["holiday_name"] = calendar["holiday_name"].str.lower()  # Convert to lowercase
calendar["holiday_name"] = calendar["holiday_name"].str.replace(" ", "_")  # Remove spaces

# Step 2: Get unique holiday names (excluding None)
unique_holidays = calendar["holiday_name"].dropna().unique()

# Step 3: Create a separate column for each holiday based on the 'holiday' column
for hld in unique_holidays:
    hld_column = []

    # Loop through each row in the 'holiday_name' column
    for index, row in calendar.iterrows():
        # Check if the current row's holiday_name matches the holiday, and if the holiday column is 1
        if row['holiday_name'] == hld and row['holiday'] == 1:
            hld_column.append(1)  # Mark as holiday
        else:
            hld_column.append(0)  # Mark as not a holiday

    # Assign the new column to the DataFrame
    calendar[hld] = hld_column

print(calendar.columns)

Index(['date', 'holiday_name', 'holiday', 'shops_closed',
       'winter_school_holidays', 'school_holidays', 'warehouse', 'year',
       'month', 'day', 'day_of_week', 'new_years_day',
       'international_womens_day', 'good_friday', 'holy_saturday',
       'easter_day', 'easter_monday', 'labour_day', 'mother's_day',
       'cyrila_a_metodej', 'jan_hus', 'den_ceske_statnosti',
       'den_vzniku_samostatneho_ceskoslovenskeho_statu',
       'den_boje_za_svobodu_a_demokracii', 'christmas_eve',
       '1st_christmas_day', '2nd_christmas_day', 'den_osvobozeni',
       'memorial_day_of_the_republic',
       'memorial_day_for_the_victims_of_the_communist_dictatorships',
       'memorial_day_for_the_victims_of_the_holocaust', 'whit_sunday',
       'whit_monday', 'national_defense_day', 'day_of_national_unity',
       'independent_hungary_day', 'state_foundation_day',
       'memorial_day_for_the_martyrs_of_arad',
       'memorial_day_of_the_1956_revolution', 'all_saints_day',
       'hungar

## 节日前后考量
节日前后也标记为holiday
如果节日放
加入boolean columns day_before_holiday 和 day_after_holiday

In [11]:
def fill_calendar2df(calendar, df_cld):
    # df_cld已经是holiday=1的数据了
    for _, row in df_cld.iterrows():
        # 店,节日名,节日的日期
        warehouse, holiday_date, holiday_name = row['warehouse'], row['date'], row['holiday_name']
        ## 劳动节和复活节是特殊节日, date_range为[-2,1],普通节日是[-1,1]
        # 在英国，复活节假期通常持续四天,欧洲劳动节貌似是1天+周末放假
        if holiday_name in ['Labour Day']:
            date_range = pd.date_range(start=holiday_date - pd.Timedelta(days=2), end=holiday_date + pd.Timedelta(days=1))
        else:  # 可能是调休,节日放假1天+周末的放假
            date_range = pd.date_range(start=holiday_date - pd.Timedelta(days=1), end=holiday_date + pd.Timedelta(days=1))
        # date_range:[-2,-1,0,1]
        for i, date in enumerate(date_range):
            mask = (calendar['warehouse'] == warehouse) & (calendar['date'] == date)
            calendar.loc[mask, 'holiday'] = 1
            # 如果不是最后一天(也就是date_range里的1),就算作holiday
            if i + 1 != len(date_range):
                calendar.loc[mask, 'holiday_name'] = holiday_name
    return calendar

# snow和precipitation由于测试集中没有,后面会drop,缺失值列就只有holiday_name了,对holiday_name进行填充‘Not’
calendar = calendar.fillna('Not')
# 转成float是为了后面特殊节日的1.5
calendar['holiday'] = calendar['holiday'].astype(float)
calendar = fill_calendar2df(calendar, calendar)
calendar.head()

# 复活节日期列表
datesx = ['03/31/2024', '04/09/2023', '04/17/2022', '04/04/2021', '04/12/2020']
# 时间字符串格式化得到复活节前1天
holidaysx = [datetime.strptime(date, '%m/%d/%Y') - timedelta(days=1) for date in datesx]
# 这3家店复活节前1天,标记为holiday
warehouses = ['Prague_1', 'Prague_2', 'Prague_3']
calendar.loc[(calendar['date'].isin(holidaysx)) & (calendar['warehouse'].isin(warehouses)), 'holiday'] = 1

# 构造一天前是不是holiday,一天后是不是holiday的特征,fillna(-1)和普通的0进行区别
calendar['day_before_holiday'] = calendar['holiday'].shift(-1).fillna(-1)
calendar['day_after_holiday'] = calendar['holiday'].shift().fillna(-1)

#拜拜了您内
calendar = calendar.drop(columns=['holiday'])

In [12]:
#检查column名字数量都没问题,导出calendar部分
print(calendar.columns)
calendar.to_csv('calendar_processed.csv', index=False)

Index(['date', 'holiday_name', 'shops_closed', 'winter_school_holidays',
       'school_holidays', 'warehouse', 'year', 'month', 'day', 'day_of_week',
       'new_years_day', 'international_womens_day', 'good_friday',
       'holy_saturday', 'easter_day', 'easter_monday', 'labour_day',
       'mother's_day', 'cyrila_a_metodej', 'jan_hus', 'den_ceske_statnosti',
       'den_vzniku_samostatneho_ceskoslovenskeho_statu',
       'den_boje_za_svobodu_a_demokracii', 'christmas_eve',
       '1st_christmas_day', '2nd_christmas_day', 'den_osvobozeni',
       'memorial_day_of_the_republic',
       'memorial_day_for_the_victims_of_the_communist_dictatorships',
       'memorial_day_for_the_victims_of_the_holocaust', 'whit_sunday',
       'whit_monday', 'national_defense_day', 'day_of_national_unity',
       'independent_hungary_day', 'state_foundation_day',
       'memorial_day_for_the_martyrs_of_arad',
       'memorial_day_of_the_1956_revolution', 'all_saints_day',
       'hungary_national_day_hol